# 7. Confidence, grounding e revisione umana

Analizza confidence e riferimenti alle evidenze della nota spese.

## 1. Import e configurazione

Carica librerie, variabili di ambiente, schema e documento.

In [ ]:
import json
import os
from pathlib import Path
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import ContentAnalyzer
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

In [ ]:
load_dotenv(r"..\.env", override=True)


In [ ]:
document_bytes = Path(r"..\file\esercizio4.pdf").read_bytes()


In [ ]:
# Legge la definizione JSON, non il risultato di un'analisi.
definizione = json.loads(
    Path(r"..\modelli\nota_spese.json").read_text(encoding="utf-8")
)


## 2. Analisi

Crea l'analyzer temporaneo, analizza il PDF e rimuove l'analyzer.

In [ ]:
credential = DefaultAzureCredential()
client = ContentUnderstandingClient(
    endpoint=os.environ["FOUNDRY_SERVICES_ENDPOINT"],
    credential=credential,
)


In [ ]:
# Crea l'analyzer dal JSON; allow_replace aggiorna l'ID del laboratorio.
try:
    client.begin_create_analyzer(
        analyzer_id="spese_confidence",
        resource=ContentAnalyzer(definizione),
        allow_replace=True,
    ).result()
except Exception:
    client.close()
    credential.close()
    raise


In [ ]:
# Analizza i byte del PDF con l'analyzer indicato e attende il risultato.
try:
    result = client.begin_analyze_binary(
        analyzer_id="spese_confidence",
        binary_input=document_bytes,
    ).result()
finally:
    try:
        # Elimina l'analyzer temporaneo identificato da analyzer_id.
        client.delete_analyzer(analyzer_id="spese_confidence")
    finally:
        client.close()
        credential.close()


## 3. Confidence e grounding

Mostra valore, confidence e origine di ogni campo.

In [ ]:
fields = result.as_dict()["contents"][0]["fields"]
value_keys = {
    "Dipendente": "valueString",
    "Progetto": "valueString",
    "TotaleSpese": "valueNumber",
    "RimborsoDovuto": "valueNumber",
    "StatoApprovazione": "valueString",
    "Sintesi": "valueString",
}

evidenze = [
    {
        "campo": name,
        "valore": fields[name][value_key],
        "confidence": fields[name]["confidence"],
        "source": fields[name]["source"],
        "spans": fields[name]["spans"],
    }
    for name, value_key in value_keys.items()
]
print(json.dumps(evidenze, ensure_ascii=False, indent=2))

## 4. Instradamento

Associa ogni campo a elaborazione automatica, revisione o scarto.

In [ ]:
decisioni = [
    {
        "campo": evidenza["campo"],
        "confidence": evidenza["confidence"],
        "instradamento": (
            "AUTOMATICO"
            if evidenza["confidence"] >= 0.90
            else "REVISIONE UMANA"
            if evidenza["confidence"] >= 0.70
            else "SCARTO"
        ),
    }
    for evidenza in evidenze
]
print(json.dumps(decisioni, ensure_ascii=False, indent=2))